# Profit Prompts — weekly voiceover batch

Turns a week of reel **plans** into voiceover audio plus word-level caption
timings, in the exact shape `reels/src/types.ts` expects.

**Runtime → Change runtime type → T4 GPU** before running anything.

### Engine
**Chatterbox** (Resemble AI) — MIT licensed, zero-shot cloning from ~10s of
reference audio, and a plain `pip install`. That last part matters: this
notebook previously used CosyVoice, whose git-clone-plus-submodules install
needed five rounds of manual dependency fixes and then produced unintelligible
audio anyway. Chatterbox is one package.

MIT is the licence that makes this usable: the page is monetised, and XTTS v2
(CPML), F5-TTS (CC-BY-NC) and Fish Speech's weights (CC-BY-NC-SA) are all
non-commercial. Licences travel with the weights, so free GPU does not fix them.

If cloning misbehaves, **cell 10 is a Gemini TTS fallback** — a prebuilt voice
rather than yours, but it is one API call and it works. Nothing downstream
cares which engine produced the MP3.

### Why a notebook and not a cron job
Colab's terms forbid bypassing the notebook interface and driving it headlessly.
Running this by hand once a week is inside the rules; wiring it into GitHub
Actions is not. The rest of the pipeline stays automated.

### Run order
1 GPU check → 2 install (**restart if asked**) → 3 plans → 4 upload reference →
5 load model → 6 generate → 7 align → 8 listen → 9 download → commit.

In [ ]:
# 1 — GPU check. Stop here if this says CPU.
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(out or "NO GPU — Runtime > Change runtime type > T4 GPU, then rerun.")

In [ ]:
# 2 — Install. Two packages, both of which can replace what Colab ships.
#
# transformers is pinned deliberately. Colab tracks the latest release, and
# Chatterbox's bundled Llama code calls the older create_causal_mask signature:
#     TypeError: create_causal_mask() got an unexpected keyword argument
#                'cache_position'
# 4.46.3 is the known-good pin; 4.51.3 also works if that one conflicts.
#
# Either package can also swap torch. When that happens the already-imported
# torch stays resident and CUDA calls fail with:
#     RuntimeError: cuDNN error: CUDNN_STATUS_SUBLIBRARY_VERSION_MISMATCH
# So this cell records versions before and after and demands a restart if
# anything moved, rather than letting you hit it three cells later.
import importlib.metadata as md

def ver(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

before = {p: ver(p) for p in ("torch", "transformers")}

!pip install -q chatterbox-tts soundfile openai-whisper
!pip install -q "transformers==4.46.3"

after = {p: ver(p) for p in ("torch", "transformers")}

changed = [p for p in before if before[p] != after[p]]
for p in before:
    print(f"{p:14} {before[p]} -> {after[p]}")

if changed:
    print("\n" + "=" * 66)
    print("  RESTART REQUIRED — changed:", ", ".join(changed))
    print("  Runtime > Restart session   (NOT 'delete runtime' — that wipes")
    print("  /content and you would have to re-upload the reference).")
    print("  Then continue from cell 3. Do NOT re-run this cell.")
    print("=" * 66)
else:
    print("\nnothing changed — continue to cell 3")

In [ ]:
# 3 — Config + load this week's plans straight from the repo.
#
# A plan is what the writer produces: {id, vo, beats, handle}. This notebook
# adds audio, captions and duration, and emits the finished reel.json.
import json, os, urllib.request
from pathlib import Path

REPO = "vaibhav018/MemeFactory"
BRANCH = "main"
WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
(WORK / "audio").mkdir(exist_ok=True)
(WORK / "reels").mkdir(exist_ok=True)

# Plans to voice this week, without the .plan.json suffix.
PLAN_IDS = ["example"]

plans = []
for pid in PLAN_IDS:
    url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/reels/data/plans/{pid}.plan.json"
    try:
        with urllib.request.urlopen(url) as r:
            plans.append(json.loads(r.read().decode()))
        print("  loaded", pid)
    except Exception as e:
        print(f"  FAILED {pid}: {e}")

print("\nplans ready:", len(plans))
for p in plans:
    print(" ", p["id"], "|", len(p["vo"].split()), "words |", len(p["beats"]), "beats")

In [ ]:
# 4 — Upload your voice reference (voice_reference_48k.wav).
#
# Kept out of the repo on purpose: it is your voice, and biometric material
# does not belong in a public repository. It survives a session restart, so
# you only upload once per runtime.
from google.colab import files

REF = WORK / "reference.wav"
if REF.exists():
    print("already uploaded:", REF, REF.stat().st_size // 1024, "KB")
else:
    up = files.upload()
    name = list(up)[0]
    REF.write_bytes(up[name])
    print("saved", name, "->", REF, REF.stat().st_size // 1024, "KB")

In [ ]:
# 5 — Build a short reference clip and load Chatterbox.
#
# Chatterbox clones from about 5-10 seconds. Feeding it the whole 2+ minute
# recording is worse, not better — an over-long prompt makes the model drift
# toward continuing the reference instead of reading your script.
import soundfile as sf, numpy as np
from chatterbox.tts import ChatterboxTTS

REF_SECONDS = 10

data, sr = sf.read(str(REF), dtype="float32", always_2d=True)
clip = data[: sr * REF_SECONDS].mean(axis=1)
REF_CLIP = WORK / f"reference_{REF_SECONDS}s.wav"
sf.write(str(REF_CLIP), clip, sr)
print("reference clip:", round(len(clip) / sr, 1), "s @", sr, "Hz")

try:
    model = ChatterboxTTS.from_pretrained(device="cuda")
    DEVICE = "cuda"
except Exception as e:
    print("\ncuda failed:", type(e).__name__, str(e)[:120])
    print("falling back to CPU — slower (minutes per clip) but reliable")
    model = ChatterboxTTS.from_pretrained(device="cpu")
    DEVICE = "cpu"

print("Chatterbox ready on", DEVICE, "| sample rate", model.sr)

In [ ]:
# 6 — Generate the voiceovers.
#
# exaggeration: emotional intensity. 0.4 reads calm and explanatory.
# cfg_weight:   lower is slower and more deliberate, which reads authoritative.
#
# The words-per-minute figure is the sanity check that matters. Natural speech
# is 140-180 wpm. Anything near 60 means the model is vocalising filler rather
# than reading the script — that is a failure, not a slow take.
import soundfile as sf

EXAGGERATION = 0.4
CFG_WEIGHT = 0.4

for plan in plans:
    wav = model.generate(plan["vo"], audio_prompt_path=str(REF_CLIP),
                         exaggeration=EXAGGERATION, cfg_weight=CFG_WEIGHT)
    arr = wav.squeeze(0).detach().cpu().numpy()
    dest = WORK / "audio" / (plan["id"] + ".wav")
    sf.write(str(dest), arr, model.sr)
    plan["_audio"] = dest
    plan["_seconds"] = len(arr) / model.sr
    words = len(plan["vo"].split())
    wpm = round(words / max(plan["_seconds"], 0.1) * 60)
    flag = "  <-- SUSPECT, listen before continuing" if wpm < 90 else ""
    print(plan["id"], "|", round(plan["_seconds"], 1), "s |", words, "words |", wpm, "wpm", flag)

In [ ]:
# 7 — Word-level alignment, MP3 encode, and the finished reel.json.
#
# Whisper runs over the GENERATED audio, not the script, so timings describe
# what was actually said. Captions drift otherwise.
#
# MP3 because CI needs this audio: ~360 KB against ~8 MB per reel as WAV is
# the difference between a repo that stays clonable and one that does not.
import subprocess, whisper, json

if "asr" not in globals():
    asr = whisper.load_model("small")

for plan in plans:
    res = asr.transcribe(str(plan["_audio"]), language="en", word_timestamps=True)
    captions = [
        {"word": w["word"].strip(), "start": round(w["start"], 3), "end": round(w["end"], 3)}
        for seg in res["segments"] for w in seg.get("words", [])
        if w["word"].strip()
    ]

    mp3 = plan["_audio"].with_suffix(".mp3")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(plan["_audio"]),
                    "-ac", "1", "-b:a", "64k", str(mp3)], check=True)
    plan["_mp3"] = mp3

    reel = {
        "id": plan["id"],
        "audioSrc": "audio/" + plan["id"] + ".mp3",
        "durationInSeconds": round(plan["_seconds"], 2),
        "handle": plan.get("handle", "@profit_prompts_"),
        "brollSrc": None,
        "captions": captions,
        "beats": plan["beats"],
    }
    (WORK / "reels" / (plan["id"] + ".reel.json")).write_text(
        json.dumps(reel, indent=2, ensure_ascii=False), encoding="utf-8")

    print(plan["id"], "|", len(captions), "words |", reel["durationInSeconds"],
          "s | mp3", mp3.stat().st_size // 1024, "KB")

    # Beats are timed by hand in the plan, so they can fall outside the audio.
    # That only shows up in a finished render otherwise.
    last = max(b["at"] for b in plan["beats"])
    if last > reel["durationInSeconds"]:
        print("   WARNING last beat at", last, "s is past the end — retime the plan")
    elif last < reel["durationInSeconds"] - 6:
        print("   NOTE last beat at", last, "s but audio runs",
              reel["durationInSeconds"], "s — long tail with no card on screen")

In [ ]:
# 8 — Listen before you ship. Bad takes are obvious and cheap to redo here.
from IPython.display import Audio, display
for plan in plans:
    print(plan["id"])
    display(Audio(str(plan["_audio"])))

In [ ]:
# 9 — Package the MP3s and reel.json files, and download.
#
# Unpack into the repo so the layout matches what Remotion expects:
#   reels/public/audio/<id>.mp3
#   reels/data/<id>.reel.json
# Both get committed. The intermediate WAVs stay here.
import shutil, os
from pathlib import Path
from google.colab import files

out = Path("/content/voiceover_batch")
if out.exists():
    shutil.rmtree(out)
(out / "public" / "audio").mkdir(parents=True)
(out / "data").mkdir(parents=True)

for plan in plans:
    shutil.copy(plan["_mp3"], out / "public" / "audio" / plan["_mp3"].name)
    shutil.copy(WORK / "reels" / (plan["id"] + ".reel.json"),
                out / "data" / (plan["id"] + ".reel.json"))

archive = shutil.make_archive("/content/voiceover_batch", "zip", out)
print(archive, os.path.getsize(archive) // 1024, "KB")
for f in sorted(out.rglob("*")):
    if f.is_file():
        print("  ", f.relative_to(out), f.stat().st_size // 1024, "KB")
files.download(archive)

In [ ]:
# 10 — FALLBACK: Gemini TTS. Only run this if cloning will not cooperate.
#
# A prebuilt voice, not yours — Gemini TTS has no cloning at all. But it is
# one API call on the free tier, and the pipeline does not care which engine
# made the MP3. Ship on this, swap your voice in later.
#
# Run this INSTEAD of cells 5 and 6, then continue at cell 7.
!pip install -q google-genai

import wave, getpass
from google import genai
from google.genai import types

client = genai.Client(api_key=getpass.getpass("Gemini API key: "))

tts = [m.name for m in client.models.list() if "tts" in m.name.lower()]
print("available:", tts)
MODEL = next((m for m in tts if "flash" in m), tts[0])
VOICE = "Charon"

# Gemini TTS is steerable in natural language. This is what closes most of the
# gap between generic TTS and something that sounds like a person.
STYLE = ("Read in a calm, confident Indian English accent. Conversational, "
         "like explaining something to a friend. Measured pace, natural pauses, "
         "slight emphasis on numbers. Not announcer-like:\n\n")

def synth(text, dest):
    r = client.models.generate_content(
        model=MODEL, contents=STYLE + text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=VOICE))),
        ),
    )
    c = r.candidates[0]
    if c.content is None or not getattr(c.content, "parts", None):
        raise RuntimeError(f"no audio returned, finish_reason={getattr(c, 'finish_reason', None)}")
    pcm = c.content.parts[0].inline_data.data
    with wave.open(str(dest), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(24000)
        w.writeframes(pcm)
    return len(pcm) / 2 / 24000

for plan in plans:
    dest = WORK / "audio" / (plan["id"] + ".wav")
    plan["_seconds"] = synth(plan["vo"], dest)
    plan["_audio"] = dest
    words = len(plan["vo"].split())
    print(plan["id"], "|", round(plan["_seconds"], 1), "s |",
          round(words / max(plan["_seconds"], 0.1) * 60), "wpm")

## After downloading

```bash
unzip voiceover_batch.zip -d reels/
cd reels
npx remotion render Reel out/<id>.mp4 --props=data/<id>.reel.json
```

Watch it once, then commit **both** the `.mp3` and the `.reel.json` — the render
job in CI needs the audio. `.gitignore` allows `reels/public/audio/*.mp3` and
blocks `*.wav`, which is also what keeps your voice reference out of the repo.

`profit_prompts_reel.yml` then picks it up at 13:00 UTC (18:30 IST).

### Weekly rhythm

| When | What |
|---|---|
| Sunday | Write next week's plans, run this notebook, commit the batch |
| Daily | Cron renders and posts from the committed `reel.json` |

Scripts have to exist a few days ahead — the one thing this free path costs
against a paid TTS API. Anything genuinely time-sensitive still goes out as a
same-day carousel.

### Troubleshooting

| Symptom | Fix |
|---|---|
| `cuDNN error: CUDNN_STATUS_SUBLIBRARY_VERSION_MISMATCH` | torch was replaced during install. **Runtime → Restart session**, then continue from cell 3. Cell 2 warns when this will happen |
| Cell 5 falls back to CPU | Expected after a cuDNN problem. Works, just minutes per clip instead of seconds |
| Output is babble in the right voice, wpm under 90 | The model is vocalising filler, not reading the script. Try a shorter reference clip (`REF_SECONDS = 5`); if it persists, use cell 10 |
| Voice sounds unlike you | Reference is noisy, or too long. 5–10 s of clean speech beats two minutes of anything |
| Delivery too flat or too theatrical | Tune `EXAGGERATION` (0.3–0.7) and `CFG_WEIGHT` (lower = slower) in cell 6 |
| Captions drift from the audio | Whisper must run on `plan["_audio"]`, not the script text |
| Beats land at the wrong moment | Retime `at` in the plan; cell 7 warns on the obvious cases |
| `ffmpeg: not found` | `!apt-get install -y ffmpeg` |

### Two rules this notebook learned the hard way

1. **Never pipe `pip install` through `tail` or `grep`** in a way that can hide a
   failure. A silent partial install is indistinguishable from a good one until
   it surfaces cells later as a confusing import error.
2. **Check words-per-minute, not just that a file appeared.** An earlier engine
   produced a 54-second file for 20 seconds of script — correct voice, no words.
   Duration against word count catches that instantly; "it generated audio"
   does not.